# [TIL] FastAPI와 SQLAlchemy ORM으로 데이터 조회하고 Swagger UI 테스트하기

**2026.11.04 (수) | KANT AX 2기**

## 1. 오늘 배운 내용

오늘은 FastAPI에서 데이터베이스를 다루는 방법을 학습했다.

기존에는 SQL문을 직접 작성해서 PostgreSQL 데이터를 조회했다면, 오늘은 **SQLAlchemy ORM**을 이용해 Python 코드로 데이터베이스에 접근하는 방법을 공부했다.

또한 Swagger UI를 활용해 API를 직접 실행하고 응답 결과를 확인하는 실습도 진행했다.

**오늘의 핵심 키워드**
- SQLAlchemy ORM
- Model / Table / Session
- SELECT와 WHERE 조건
- 단일 회원 조회
- None과 404 Not Found
- Swagger UI
- GET과 POST 요청의 차이

---

## 2. SQLAlchemy ORM이란?

ORM(Object Relational Mapping)은 데이터베이스의 테이블을 Python 객체와 연결해 주는 기술이다.

SQL문을 직접 작성하지 않고 Python 코드를 이용해 데이터를 조회하거나 관리할 수 있다.

### 기존 SQL 방식

```sql
SELECT member_id, name, email, age
FROM members
WHERE member_id = 1;
```

### SQLAlchemy ORM 방식

```python
stmt = select(Member).where(
    Member.member_id == 1
)

member = session.scalar(stmt)

print(member)
```

두 코드는 모두 `member_id`가 1인 회원을 조회한다.

SQL에서는 `WHERE` 조건을 사용하고, ORM에서는 Python의 `where()` 메서드를 사용한다.

**오늘 이해한 점**

ORM을 사용한다고 SQL을 몰라도 되는 것은 아니다. ORM 내부에서도 SQL이 실행되므로, SQL의 기본 개념을 알고 있어야 코드를 이해하기 쉽다.

---

## 3. Model, Table, Session 이해하기

SQLAlchemy ORM을 이해하려면 세 가지 개념이 중요하다.

| 개념 | 역할 |
|---|---|
| Model | 데이터베이스 테이블에 대응하는 Python 클래스 |
| Table | 실제 데이터가 저장되는 데이터베이스 테이블 |
| Session | 데이터베이스 조회와 변경 작업을 처리하는 객체 |

예를 들어 `Member`라는 모델을 이용하면 데이터베이스의 `members` 테이블을 Python 코드로 다룰 수 있다.

Session은 데이터베이스에 작업을 요청하고 결과를 받아오는 역할을 한다.

---

## 4. 특정 회원 조회하기

오늘 진행한 실습에서는 특정 회원의 ID를 이용해 데이터를 조회했다.

```python
stmt = select(Member).where(
    Member.member_id == 1
)

member = session.scalar(stmt)

print(member)
```

**코드 분석**

1. `select(Member)` : Member 모델의 데이터를 조회한다.
2. `.where(Member.member_id == 1)` : 회원 ID가 1인 데이터를 찾는다.
3. `session.scalar(stmt)` : 조회 결과에서 첫 번째 행의 ORM 객체를 가져온다.
4. `print(member)` : 조회한 결과를 출력한다.

SQL의 SELECT와 WHERE를 Python 코드로 표현할 수 있다는 점이 인상적이었다.

---

## 5. 존재하지 않는 회원 조회하기

이번에는 존재하지 않는 회원 ID로 조회하는 실습을 진행했다.

```python
stmt = select(Member).where(
    Member.member_id == 999999
)

member = session.scalar(stmt)

print(member)
```

**실행 결과**

```text
None
```

조회 조건에 맞는 데이터가 없다면 `None`이 반환된다.

여기서 중요한 사실을 배웠다.

**데이터베이스에서 데이터가 없는 것과 HTTP 404 응답은 서로 다른 개념이다.**

- ORM 조회 결과가 없다면 `None`을 반환할 수 있다.
- FastAPI에서는 이 결과를 확인한 뒤 `404 Not Found` 응답으로 처리할 수 있다.

즉, 데이터 조회와 HTTP 응답 처리는 구분해서 생각해야 한다.

---

## 6. Swagger UI로 API 확인하기

Python 코드로 데이터를 조회하는 것에서 끝나지 않고, Swagger UI에서 API를 직접 테스트하는 방법도 학습했다.

FastAPI는 API 문서를 자동으로 생성해 주기 때문에 브라우저에서 요청을 쉽게 실행할 수 있다.

### Swagger 접속

서버를 실행한 뒤 다음 주소로 접속한다.

```text
http://127.0.0.1:8010/docs
```

### 테스트 과정

1. Swagger UI에 접속한다.
2. 테스트할 GET API를 선택한다.
3. `Try it out` 버튼을 클릭한다.
4. 회원 ID 등의 파라미터를 입력한다.
5. `Execute` 버튼을 클릭한다.
6. Response body와 Response code를 확인한다.

### 확인할 응답 코드

| 코드 | 의미 |
|---|---|
| 200 OK | 요청 성공 |
| 201 Created | 새로운 리소스 생성 성공 |
| 404 Not Found | 요청한 리소스를 찾을 수 없음 |
| 422 Unprocessable Content | 요청 데이터 검증 실패 |

Swagger는 단순히 API 목록을 보여주는 문서가 아니라, **API가 실제로 어떻게 동작하는지 확인할 수 있는 테스트 도구**라는 점을 배웠다.

---

## 7. GET과 POST 요청의 차이

실습 중 GET과 POST의 데이터 크기 제한에 대해서도 공부했다.

| 구분 | GET | POST |
|---|---|---|
| 주요 목적 | 데이터 조회 | 데이터 전송 및 생성 |
| 데이터 전달 | 주로 URL의 경로·쿼리 | 주로 요청 본문(Body) |
| 데이터 크기 | URL 길이에 영향 | 서버 설정 등에 영향 |
| Swagger 테스트 | Parameters | Request body |

### GET 요청

```http
GET /members/1
```

특정 회원의 데이터를 조회하는 데 사용한다.

### POST 요청

```http
POST /members
```

```json
{
  "name": "Park",
  "email": "park@example.com",
  "age": 28
}
```

새로운 회원 정보를 등록하는 데 사용할 수 있다.

**새롭게 알게 된 점**

GET과 POST의 최대 데이터 크기는 HTTP 메서드 자체에 일률적으로 정해진 값이 있는 것이 아니다.

GET은 브라우저나 서버의 URL 길이 제한을 고려해야 하고, POST는 웹 서버와 애플리케이션의 요청 본문 크기 제한을 고려해야 한다.

또한 POST라고 해서 자동으로 안전한 것은 아니며, 민감한 정보는 HTTPS 등을 이용해 보호해야 한다.

---

## 8. 오늘 배운 내용 정리

오늘은 데이터베이스 조회부터 API 테스트까지 연결되는 흐름을 공부했다.

```text
사용자
  ↓
Swagger UI
  ↓
FastAPI 엔드포인트
  ↓
SQLAlchemy ORM
  ↓
PostgreSQL
  ↓
조회 결과 반환
  ↓
FastAPI HTTP 응답
  ↓
Swagger UI에서 결과 확인
```

처음에는 SQL과 ORM이 별개의 기술처럼 느껴졌지만, 실습하면서 결국 같은 데이터베이스 작업을 서로 다른 방식으로 표현한다는 점을 이해할 수 있었다.

특히 `None`과 `404 Not Found`의 차이를 공부하면서 **데이터베이스 처리와 API 응답 처리는 서로 구분해야 한다**는 사실을 알게 되었다.

## 9. 오늘의 회고

FastAPI를 처음 배울 때는 API를 실행하는 것만으로도 신기했는데, 이제는 PostgreSQL에 저장된 데이터를 Python ORM으로 조회하고 Swagger UI를 통해 확인하는 과정까지 배우고 있다.

아직 Model, Session, HTTP 응답 코드 등 익숙하지 않은 개념들이 있지만, 하나씩 실습하면서 전체적인 구조를 이해해 가고 있다.

오늘 가장 중요하게 느낀 것은 단순히 코드를 실행하는 것이 아니라 **요청이 들어온 후 데이터베이스를 거쳐 응답이 반환되기까지의 흐름을 이해하는 것**이다.

앞으로는 회원 등록, 조회뿐만 아니라 수정(PUT/PATCH), 삭제(DELETE) 기능까지 학습하면서 CRUD API의 전체 구조를 이해하는 것이 목표다.

**오늘의 한 줄 정리**

> SQLAlchemy ORM으로 데이터를 조회하고, FastAPI와 Swagger UI를 통해 API의 동작과 HTTP 응답을 확인하는 방법을 배웠다.

#TIL #FastAPI #SQLAlchemy #SwaggerUI #백엔드개발